<a href="https://colab.research.google.com/github/your-org/alexpose/blob/main/experiments/multiple-sclerosis/01_pose_extraction_from_raw_video.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 - Pose extraction from raw video

A model cannot learn from pixels here; it learns from **skeletons**. In this notebook we run MediaPipe BlazePose over each video and get 33 body landmarks per frame. We reuse the existing `alexpose` pose code, so there is no new pose logic to trust, just a thin loop that feeds whole frames to the detector.

The result for each video is an array of shape `(T, 33, 3)`: `T` frames, 33 joints, and three numbers per joint (x, y in pixels, plus a visibility score). We clean it, normalize it, and cache it so every later notebook opens instantly.


In [ ]:
# --- Setup: install dependencies (Colab installs; local usually already has them) ---
import importlib, importlib.util, subprocess, sys, os

IN_COLAB = 'google.colab' in sys.modules

def _need(mod):
    return importlib.util.find_spec(mod) is None

# Light deps used by every notebook.
_pkgs = []
for mod, pip_name in [('cv2','opencv-python'), ('mediapipe','mediapipe'),
                      ('sklearn','scikit-learn'), ('pandas','pandas'),
                      ('matplotlib','matplotlib'), ('tqdm','tqdm')]:
    if _need(mod):
        _pkgs.append(pip_name)
if _pkgs:
    print('installing:', _pkgs)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *_pkgs])
else:
    print('all light dependencies already present')

In [ ]:
# --- Make `sjepa` and `ambient` importable, locally and in Colab ---
from pathlib import Path
import sys, subprocess

def _find_exp_dir():
    # Local run: this notebook sits in experiments/multiple-sclerosis.
    here = Path.cwd()
    for p in [here, *here.parents]:
        if (p / 'sjepa' / '__init__.py').exists():
            return p
    return None

EXP_DIR = _find_exp_dir()
if EXP_DIR is None:
    # Colab: clone the repo, then point at the experiment folder.
    REPO = 'https://github.com/your-org/alexpose.git'  # <-- edit to your fork
    if not Path('alexpose').exists():
        subprocess.check_call(['git', 'clone', '--depth', '1', REPO])
    EXP_DIR = Path('alexpose') / 'experiments' / 'multiple-sclerosis'

REPO_ROOT = EXP_DIR.parents[1]
for p in (str(EXP_DIR), str(REPO_ROOT)):
    if p not in sys.path:
        sys.path.insert(0, p)
print('experiment dir:', EXP_DIR)
print('repo root     :', REPO_ROOT)

In [ ]:
# --- Paths and profile (reads the root .env if python-dotenv is present) ---
import os
try:
    from dotenv import load_dotenv
    load_dotenv(REPO_ROOT / '.env')
except Exception:
    pass

VIDEO_DIR = EXP_DIR / 'video-data'
ARTIFACT_DIR = EXP_DIR / 'artifacts'
KEYPOINTS_DIR = ARTIFACT_DIR / 'keypoints'
IMAGES_DIR = EXP_DIR / 'images'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# Pick the model size profile. 'laptop' is the fast default; set SJEPA_PROFILE=gpu
# in your .env for a larger model, or SJEPA_SMOKE=1 for a near-instant test run.
os.environ.setdefault('SJEPA_PROFILE', 'laptop')
print('SJEPA_PROFILE =', os.environ['SJEPA_PROFILE'],
      '| SJEPA_SMOKE =', os.environ.get('SJEPA_SMOKE', '0'))

## One frame at a time

The loader opens a video, samples it down to about 15 frames per second, and asks MediaPipe for the pose in each sampled frame. Unlike the GAVD pipeline it needs no bounding boxes and no annotation CSVs; it just reads the whole frame.


In [ ]:
from ambient.pose.model_management import MediaPipeModelManager
from ambient.pose.keypoint_extractor import SequenceKeypointExtractor
from sjepa.data import load_video_sequence, clean_sequence, normalize_sequence

MediaPipeModelManager().ensure_model_available()  # downloads the model once
extractor = SequenceKeypointExtractor()

sample = sorted((VIDEO_DIR / 'normal').glob('*.mp4'))[0]
seq = load_video_sequence(sample, target_fps=15, extractor=extractor, verbose=True)
print('raw sequence shape:', seq.shape, '  (frames, joints, [x, y, visibility])')

## Clean and normalize

Real videos have frames where the detector loses the person. We interpolate short gaps and drop videos that are mostly empty. Then we **normalize**: we move the pelvis to the origin and scale by the torso length. This removes where the walker stood and how close the camera was, so the model sees the shape of the motion rather than the framing.


In [ ]:
cleaned = clean_sequence(seq)
normalized = normalize_sequence(cleaned)
print('cleaned:', cleaned.shape, '| normalized:', normalized.shape)
print('normalized x range:', round(float(normalized[:,:,0].min()),2),
      'to', round(float(normalized[:,:,0].max()),2))

## See the skeleton move

Here is the skeleton the model will actually train on. The animation draws the BlazePose stick figure over time. This is the same view we will reuse in notebook 02 to show the mask.


In [ ]:
from sjepa.viz import skeleton_animation
from IPython.display import Image

gif = skeleton_animation(normalized, ARTIFACT_DIR / 'demo_skeleton.gif',
                         fps=15, title='normal gait (normalized)')
Image(filename=str(gif))

## Extract and cache every video

Now we run the same steps over all clips and cache one `.npz` file per video. This is the slow step, a few minutes on a laptop, and only needs to run once. If the cache already exists (it ships with the repo) this loop just confirms it.


In [ ]:
from sjepa.data import save_sequence_npz, source_id_from_name
import numpy as np

KEYPOINTS_DIR.mkdir(parents=True, exist_ok=True)
index = []
for label in ['normal', 'ms', 'pd']:
    for vid in sorted((VIDEO_DIR / label).glob('*.mp4')):
        sid = source_id_from_name(vid.name)
        out = KEYPOINTS_DIR / f'{label}__{sid}__{vid.stem}.npz'
        if out.exists():
            with np.load(out, allow_pickle=True) as z:
                n = int(z['keypoints_norm'].shape[0])
        else:
            raw = load_video_sequence(vid, target_fps=15, extractor=extractor)
            cl = clean_sequence(raw)
            if cl is None or cl.shape[0] < 8:
                print('skip (too few valid frames):', vid.name); continue
            nm = normalize_sequence(cl)
            save_sequence_npz(out, cl, nm, 15, sid, label, vid.stem)
            n = nm.shape[0]
        index.append(dict(label=label, source_id=sid, clip_name=vid.stem, n_frames=n))

import pandas as pd
idx = pd.DataFrame(index)
idx.to_parquet(ARTIFACT_DIR / 'keypoints_index.parquet', index=False)
print(idx.groupby('label').agg(videos=('clip_name','count'),
                               sources=('source_id','nunique'),
                               frames=('n_frames','sum')))

### Quick checks

A good habit: assert the shapes and ranges are what we expect before moving on.


In [ ]:
from sjepa.data import load_index
recs = load_index(KEYPOINTS_DIR)
assert len(recs) > 0
for r in recs[:5]:
    a = r.load_norm()
    assert a.ndim == 3 and a.shape[1] == 33 and a.shape[2] == 3
print(f'cached {len(recs)} sequences, all shaped (T, 33, 3). Ready for notebook 02.')